# Person 2 - Step 1: check the shared evaluation script

Run this **after** you have uploaded the Step 1 files to GitHub. It downloads the repo fresh,
so it tests exactly what your team will get. No GPU needed (Runtime type: CPU is fine). About 2 minutes.

What each cell proves:
1. the repo downloads and the packages install
2. all automated tests pass (Person 3's 23 + 8 new evaluation tests)
3. the "dumb" systems give the expected numbers with the NEW official scorer
4. the evaluation script scores a real system (Assignment 1 Baseline B) end to end
5. the comparison test works (Section 10.2)

In [ ]:
# Cell 1: fresh copy of the team repo + packages
%cd /content
!rm -rf SenseShift
!git clone -q https://github.com/abiha25/SenseShift-Semantic-change-across-multiple-time-periods.git SenseShift
%cd /content/SenseShift
!pip install -q bcubed nltk pyyaml
!git log --oneline -3

**Cell 2 - tests.** Expected last line: `31 passed`. If anything fails, stop and send me the output.

In [ ]:
# Cell 2: automated tests
!python -m pytest tests -q 2>&1 | tail -3

**Cell 3 - sanity check.** Expected: one_cluster `B3 P 0.646  B3 R 0.981  B3 F1 0.756  rho 0.000`;
gold_labels all `1.000`; Subtask 2 `all_1 = 0.236`, `all_0 = 0.361`.

(Before the organisers' update of 8 Nov 2026, all_1 was 0.472 and P/R were swapped.)

In [ ]:
# Cell 3: sanity check with the new official scorer
!python src/sanity_check.py

**Cell 4 - score a real system.** Assignment 1 Baseline B (XL-LEXEME). Expected: 1a `0.760`, 1b `0.492`
(same as Assignment 1, because these metrics did not change), and Subtask 2 `0.282` under the new macro F1
(the old metric, shown as "positive-class F1 (A1 metric)", is still `0.468`).

In [ ]:
# Cell 4: full evaluation report of a real system
!python -m src.evaluate score --name A1_baselineB_xllex \
    --s1 results/pretrained/SV_subtask1_xllex_combined.jsonl \
    --s2 results/pretrained/SV_subtask2_xllex_cos_rankthr.jsonl \
    --out /content/eval_check/B --subgroups

**Cell 5 - compare two systems** (Assignment 1 Baseline B vs Baseline A, Subtask 1a).
Expected: difference about `+0.063`, p = `0.5`, verdict *not significant* - with 10 words, a gain of
0.06 is not enough evidence. This is exactly the rule in Section 10.2.

In [ ]:
# Cell 5: is B really better than A?
!python -m src.evaluate score --name A1_baselineA --s1 results/classical/SV_subtask1_combined.jsonl \
    --s2 results/classical/SV_subtask2_LR_rank_thr_0.5.jsonl --out /content/eval_check/A > /dev/null
!python -m src.evaluate compare /content/eval_check/B/scores.json /content/eval_check/A/scores.json --metric 1a

Done. If Cells 2-5 match, tell the team the evaluation script is ready (message in the chat).